# FLUX Apex Validation — Clean Architecture Tests

**Purpose:** Validate Flux-Apex-V1.flx structure and weights with correct data access patterns

**Model:** Flux-Apex-V1.flx v8.1-complete (~17 GB, ~8.34B params)

---

## Test Categories

| Category | What We Test | Expected |
|----------|--------------|----------|
| **Structure** | Top-level keys, version, format | 27 keys, v8.1-complete |
| **Native FLUX** | CSE, Field, Memory, Causal, etc. | ~1.4B params |
| **Embedded Models** | 12 models with weights | ~6.4B params |
| **Detection** | Face, Depth, Pose, Object | ~538M params |
| **Runtime** | Embedded Python code | 87 files |
| **Orchestration** | Tools and system prompt | 8+ tools |

In [ ]:
# Cell 1: Setup
import sys
import os
import gc
import torch
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional, Tuple

# Environment detection
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
ON_COLAB = 'COLAB_GPU' in os.environ

# Set up paths
if ON_KAGGLE:
    !git clone https://github.com/Unseengap/FLUX.git /kaggle/working/flux 2>/dev/null || git -C /kaggle/working/flux pull
    ROOT = Path('/kaggle/working/flux')
elif ON_COLAB:
    !git clone https://github.com/Unseengap/FLUX.git /content/flux 2>/dev/null || git -C /content/flux pull  
    ROOT = Path('/content/flux')
else:
    ROOT = Path('.').resolve()
    if not (ROOT / 'flux_model.py').exists():
        ROOT = ROOT.parent

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print(f"Environment: {'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'Local'}")
print(f"Root: {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Download Model
from huggingface_hub import hf_hub_download

HF_REPO = "UnseenGAP/FLUX"
MODEL_PATH = ROOT / 'checkpoints' / 'Flux-Apex-V1.flx'
MODEL_PATH.parent.mkdir(exist_ok=True)

# Get HF token
HF_TOKEN = None
if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except: pass
else:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if MODEL_PATH.exists():
    print(f"✓ Model cached: {MODEL_PATH.stat().st_size / 1e9:.2f} GB")
else:
    print("📥 Downloading Flux-Apex-V1.flx...")
    hf_hub_download(
        repo_id=HF_REPO,
        filename='checkpoints/Flux-Apex-V1.flx',
        local_dir=str(ROOT),
        token=HF_TOKEN,
    )
    print(f"✓ Downloaded: {MODEL_PATH.stat().st_size / 1e9:.2f} GB")

In [ ]:
# Cell 3: Helper Functions

def count_params(obj, depth=0, max_depth=10) -> int:
    """Recursively count parameters in nested structures."""
    if depth > max_depth:
        return 0
    if isinstance(obj, torch.Tensor):
        return obj.numel()
    if isinstance(obj, dict):
        return sum(count_params(v, depth+1, max_depth) for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return sum(count_params(x, depth+1, max_depth) for x in obj)
    return 0

def count_tensors(obj, depth=0, max_depth=10) -> int:
    """Count number of tensors in nested structures."""
    if depth > max_depth:
        return 0
    if isinstance(obj, torch.Tensor):
        return 1
    if isinstance(obj, dict):
        return sum(count_tensors(v, depth+1, max_depth) for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return sum(count_tensors(x, depth+1, max_depth) for x in obj)
    return 0

def format_params(n: int) -> str:
    """Format parameter count nicely."""
    if n >= 1e9:
        return f"{n/1e9:.2f}B"
    if n >= 1e6:
        return f"{n/1e6:.1f}M"
    if n >= 1e3:
        return f"{n/1e3:.1f}K"
    return str(n)

@dataclass
class TestResult:
    name: str
    category: str 
    passed: bool
    score: float = 0.0
    threshold: float = 0.0
    details: str = ""

class ValidationReport:
    def __init__(self):
        self.results: List[TestResult] = []
        
    def add(self, result: TestResult):
        self.results.append(result)
        icon = "✅" if result.passed else "❌"
        print(f"{icon} [{result.category}] {result.name}")
        print(f"   Score: {result.score:.2f} / Threshold: {result.threshold}")
        if result.details:
            print(f"   {result.details}")
    
    def summary(self):
        passed = sum(1 for r in self.results if r.passed)
        total = len(self.results)
        print(f"\n{'='*60}")
        print(f"SUMMARY: {passed}/{total} tests passed ({100*passed/total:.1f}%)")
        print(f"{'='*60}")
        
        by_cat = {}
        for r in self.results:
            by_cat.setdefault(r.category, []).append(r)
        
        for cat, tests in by_cat.items():
            cat_passed = sum(1 for t in tests if t.passed)
            icon = "✅" if cat_passed == len(tests) else "⚠️"
            print(f"{icon} {cat}: {cat_passed}/{len(tests)}")

report = ValidationReport()
print("✓ Helpers loaded")

In [ ]:
# Cell 4: Load Model
print("Loading Flux-Apex-V1.flx...")
flx = torch.load(str(MODEL_PATH), map_location='cpu', weights_only=False)

print(f"\n{'='*60}")
print("MODEL OVERVIEW")
print(f"{'='*60}")
print(f"Format: {flx.get('format', 'N/A')}")
print(f"Version: {flx.get('version', 'N/A')}")
print(f"Phase: {flx.get('phase', 'N/A')}")
print(f"Top-level keys: {len(flx.keys())}")
print(f"Can continue learning: {flx.get('can_continue_learning', False)}")

# List top-level keys
print(f"\nTop-level keys:")
for key in sorted(flx.keys()):
    val = flx[key]
    if isinstance(val, dict):
        print(f"  {key}: dict ({len(val)} keys)")
    elif isinstance(val, torch.Tensor):
        print(f"  {key}: tensor {val.shape}")
    elif isinstance(val, (list, tuple)):
        print(f"  {key}: list ({len(val)} items)")
    else:
        print(f"  {key}: {type(val).__name__}")

In [ ]:
# Cell 5: Test Structure
print(f"\n{'='*60}")
print("TEST 1: STRUCTURE")
print(f"{'='*60}")

# Version check
version = flx.get('version', '')
report.add(TestResult(
    name="version",
    category="Structure",
    passed='8.1' in version or 'complete' in version,
    score=1.0 if '8.1' in version else 0.5,
    threshold=0.5,
    details=f"Version: {version}"
))

# Format check
fmt = flx.get('format', '')
report.add(TestResult(
    name="format",
    category="Structure",
    passed=fmt == 'FLUX',
    score=1.0 if fmt == 'FLUX' else 0.0,
    threshold=1.0,
    details=f"Format: {fmt}"
))

# Top-level keys count
key_count = len(flx.keys())
report.add(TestResult(
    name="top_level_keys",
    category="Structure",
    passed=key_count >= 20,
    score=key_count,
    threshold=20,
    details=f"{key_count} top-level keys"
))

In [ ]:
# Cell 6: Test Native Components (Correct Access Patterns)
print(f"\n{'='*60}")
print("TEST 2: NATIVE FLUX COMPONENTS")
print(f"{'='*60}")

# Expected native components with minimum parameter counts
NATIVE_COMPONENTS = {
    'cse': 1_000_000,           # ~1.3M
    'field': 25_000_000,        # ~28M (compressed)
    'memory': 500_000_000,      # ~542M
    'bridges': 400_000_000,     # ~456M
    'causal': 100_000_000,      # ~150M
    'gravity': 70_000_000,      # ~75M
    'thermodynamic': 130_000_000,  # ~135M
    'causal_wave_chaining': 500_000,  # ~570K
    'grid_to_wave': 100_000,    # ~192K
    'spatial_memory': 10_000,   # ~12K
    'adapters': 10_000_000,     # ~15M
}

native_total = 0
native_found = 0

for comp_name, min_params in NATIVE_COMPONENTS.items():
    if comp_name in flx:
        comp = flx[comp_name]
        params = count_params(comp)
        native_total += params
        
        passed = params >= min_params
        if passed:
            native_found += 1
            
        report.add(TestResult(
            name=f"{comp_name}_params",
            category="Native",
            passed=passed,
            score=params / 1e6,
            threshold=min_params / 1e6,
            details=f"{format_params(params)} (expected >= {format_params(min_params)})"
        ))
    else:
        report.add(TestResult(
            name=f"{comp_name}_present",
            category="Native",
            passed=False,
            score=0,
            threshold=1,
            details=f"Component '{comp_name}' not found"
        ))

print(f"\n📊 Native Summary: {native_found}/{len(NATIVE_COMPONENTS)} components")
print(f"📊 Native Total: {format_params(native_total)}")

In [ ]:
# Cell 7: Test Embedded Models (CORRECT ACCESS PATTERN)
print(f"\n{'='*60}")
print("TEST 3: EMBEDDED MODELS")
print(f"{'='*60}")

# Models are stored under flx['models'][name]['weights']
EXPECTED_MODELS = {
    'instruct': 1_500_000_000,   # ~1.54B
    'vision': 2_000_000_000,     # ~2.21B
    'coder': 1_500_000_000,      # ~1.54B
    'clip': 400_000_000,         # ~428M
    'tts': 400_000_000,          # ~410M
    'whisper': 200_000_000,      # ~242M
    'embedding': 20_000_000,     # ~23M
}

models_dict = flx.get('models', {})
models_total = 0
models_found = 0

print(f"\nModels dict has {len(models_dict)} entries:")
for name in models_dict:
    print(f"  - {name}")

print()

for model_name, min_params in EXPECTED_MODELS.items():
    if model_name in models_dict:
        model_data = models_dict[model_name]
        
        # Weights are nested under 'weights' key
        weights = model_data.get('weights', {})
        params = count_params(weights)
        
        # Also check total_params metadata if weights counting fails
        if params == 0:
            params = model_data.get('total_params', 0)
        
        models_total += params
        passed = params >= min_params * 0.9  # 10% tolerance
        
        if passed:
            models_found += 1
        
        base_model = model_data.get('base_model', 'N/A')
        lazy = model_data.get('lazy_load', False)
        
        report.add(TestResult(
            name=f"{model_name}_weights",
            category="Models",
            passed=passed,
            score=params / 1e9,
            threshold=min_params / 1e9,
            details=f"{format_params(params)} from {base_model[:40]} [{'lazy' if lazy else 'eager'}]"
        ))
    else:
        report.add(TestResult(
            name=f"{model_name}_present",
            category="Models",
            passed=False,
            score=0,
            threshold=1,
            details=f"Model '{model_name}' not found in models dict"
        ))

print(f"\n📊 Models Summary: {models_found}/{len(EXPECTED_MODELS)} models with weights")
print(f"📊 Models Total: {format_params(models_total)}")

In [ ]:
# Cell 8: Test Detection Models
print(f"\n{'='*60}")
print("TEST 4: DETECTION MODELS")
print(f"{'='*60}")

# Detection models are also under flx['models']
DETECTION_MODELS = {
    'depth': 300_000_000,       # ~344M
    'object_detect': 150_000_000,  # ~155M
    'pose': 30_000_000,         # ~39M
    'face': 0,                  # ONNX (params stored differently)
    'speaker_detect': 0,        # Placeholder
}

detection_total = 0
detection_found = 0

for model_name, min_params in DETECTION_MODELS.items():
    if model_name in models_dict:
        model_data = models_dict[model_name]
        
        # Check for ONNX models (face uses ONNX)
        is_onnx = model_data.get('format') == 'onnx' or 'onnx_models' in model_data
        
        if is_onnx:
            onnx_models = model_data.get('onnx_models', {})
            onnx_count = len(onnx_models)
            passed = onnx_count >= 1
            detection_found += 1 if passed else 0
            
            report.add(TestResult(
                name=f"{model_name}_onnx",
                category="Detection",
                passed=passed,
                score=onnx_count,
                threshold=1,
                details=f"{onnx_count} ONNX models: {list(onnx_models.keys())[:3]}..."
            ))
        else:
            weights = model_data.get('weights', {})
            params = count_params(weights)
            if params == 0:
                params = model_data.get('total_params', 0)
            
            detection_total += params
            passed = params >= min_params * 0.9 if min_params > 0 else True
            if passed:
                detection_found += 1
            
            base_model = model_data.get('base_model', 'N/A')
            
            report.add(TestResult(
                name=f"{model_name}_weights",
                category="Detection",
                passed=passed,
                score=params / 1e6,
                threshold=min_params / 1e6,
                details=f"{format_params(params)} from {base_model[:40]}"
            ))
    else:
        report.add(TestResult(
            name=f"{model_name}_present",
            category="Detection",
            passed=False,
            score=0,
            threshold=1,
            details=f"Detection model '{model_name}' not found"
        ))

print(f"\n📊 Detection Summary: {detection_found}/{len(DETECTION_MODELS)} models")
print(f"📊 Detection Total: {format_params(detection_total)}")

In [ ]:
# Cell 9: Test Memory Structure
print(f"\n{'='*60}")
print("TEST 5: MEMORY SYSTEM")
print(f"{'='*60}")

memory = flx.get('memory', {})

# Memory has nested structure
memory_tiers = ['working', 'episodic', 'semantic', 'router']
state_dict = memory.get('state_dict', {})

print(f"Memory keys: {list(memory.keys())}")
print(f"Memory state_dict keys: {list(state_dict.keys())}")

# Check each tier
for tier in memory_tiers:
    # Could be in state_dict as {tier}_memory_state or directly as {tier}
    tier_key = f"{tier}_memory_state" if f"{tier}_memory_state" in state_dict else tier
    tier_data = state_dict.get(tier_key) or memory.get(tier)
    
    present = tier_data is not None
    params = count_params(tier_data) if present else 0
    
    report.add(TestResult(
        name=f"memory_{tier}",
        category="Memory",
        passed=present,
        score=params / 1e6 if params > 0 else (1.0 if present else 0.0),
        threshold=0.5,
        details=f"{tier}: {'present' if present else 'missing'} ({format_params(params)})"
    ))

# Total memory params
total_memory = count_params(memory)
print(f"\n📊 Memory Total: {format_params(total_memory)}")

In [ ]:
# Cell 10: Test Field Structure (Compressed)
print(f"\n{'='*60}")
print("TEST 6: RESONANCE FIELD")
print(f"{'='*60}")

field = flx.get('field', {})
field_state = field.get('state_dict', field)
field_config = field.get('config', {})

print(f"Field keys: {list(field.keys())}")
print(f"Field state_dict keys: {list(field_state.keys()) if isinstance(field_state, dict) else 'N/A'}")
print(f"Field config: {field_config}")

# Field is compressed - check for 'compressed' key instead of 'state'
has_compressed = 'compressed' in field_state
has_projection = 'projection_matrix' in field_state
has_original = 'original_shape' in field_state

# Get compression info
if has_compressed:
    compressed_tensor = field_state['compressed']
    print(f"Compressed tensor shape: {compressed_tensor.shape}")
    print(f"Compressed tensor dtype: {compressed_tensor.dtype}")

if has_original:
    original_shape = field_state['original_shape']
    print(f"Original shape: {original_shape}")

field_params = count_params(field)

report.add(TestResult(
    name="field_compressed",
    category="Field",
    passed=has_compressed,
    score=1.0 if has_compressed else 0.0,
    threshold=1.0,
    details=f"Compressed: {has_compressed}, Projection: {has_projection}"
))

report.add(TestResult(
    name="field_params",
    category="Field",
    passed=field_params > 25_000_000,
    score=field_params / 1e6,
    threshold=25,
    details=f"{format_params(field_params)}"
))

In [ ]:
# Cell 11: Test Runtime Embed
print(f"\n{'='*60}")
print("TEST 7: EMBEDDED RUNTIME")
print(f"{'='*60}")

runtime = flx.get('runtime', {})

print(f"Runtime keys: {list(runtime.keys())}")

# Check for embedded code
code = runtime.get('code', {})
metadata = runtime.get('metadata', {})

file_count = len(code)
total_lines = metadata.get('total_lines', 0)
version = runtime.get('version', 'N/A')
has_bootstrap = 'bootstrap' in runtime

print(f"Version: {version}")
print(f"Files: {file_count}")
print(f"Lines: {total_lines}")
print(f"Bootstrap: {has_bootstrap}")

if file_count > 0:
    print(f"\nSample files:")
    for f in list(code.keys())[:10]:
        print(f"  - {f}")

report.add(TestResult(
    name="runtime_files",
    category="Runtime",
    passed=file_count >= 80,
    score=file_count,
    threshold=80,
    details=f"{file_count} Python files embedded"
))

report.add(TestResult(
    name="runtime_bootstrap",
    category="Runtime",
    passed=has_bootstrap,
    score=1.0 if has_bootstrap else 0.0,
    threshold=1.0,
    details="bootstrap.py embedded" if has_bootstrap else "No bootstrap"
))

In [ ]:
# Cell 12: Test Orchestration
print(f"\n{'='*60}")
print("TEST 8: ORCHESTRATION")
print(f"{'='*60}")

orch = flx.get('orchestration', {})

print(f"Orchestration keys: {list(orch.keys())}")

tools = orch.get('tools', [])
system_prompt = orch.get('system_prompt', '')
tool_format = orch.get('tool_format', 'N/A')

tool_count = len(tools) if isinstance(tools, list) else 0
has_prompt = len(system_prompt) > 100

print(f"Tool format: {tool_format}")
print(f"Tool count: {tool_count}")
print(f"System prompt: {len(system_prompt)} chars")

if tool_count > 0:
    print(f"\nTools:")
    for t in tools[:5]:
        if isinstance(t, dict) and 'function' in t:
            name = t['function'].get('name', 'unknown')
            print(f"  - {name}")

report.add(TestResult(
    name="orchestration_tools",
    category="Orchestration",
    passed=tool_count >= 5,
    score=tool_count,
    threshold=5,
    details=f"{tool_count} tools defined"
))

report.add(TestResult(
    name="orchestration_prompt",
    category="Orchestration",
    passed=has_prompt,
    score=len(system_prompt),
    threshold=100,
    details=f"System prompt: {len(system_prompt)} chars"
))

In [ ]:
# Cell 13: Test Causal Components
print(f"\n{'='*60}")
print("TEST 9: CAUSAL SYSTEM")
print(f"{'='*60}")

# Causal component (Phase 5 injection)
causal = flx.get('causal', {})
causal_state = causal.get('state_dict', {})

has_cgn = 'cgn_state' in causal_state
has_tl = 'tl_state' in causal_state
causal_params = count_params(causal)

print(f"Causal keys: {list(causal.keys())}")
print(f"Causal state_dict keys: {list(causal_state.keys())}")
print(f"CGN state: {has_cgn}")
print(f"TL state: {has_tl}")
print(f"Causal params: {format_params(causal_params)}")

report.add(TestResult(
    name="causal_cgn",
    category="Causal",
    passed=has_cgn,
    score=1.0 if has_cgn else 0.0,
    threshold=1.0,
    details="CGN state present" if has_cgn else "CGN state missing"
))

report.add(TestResult(
    name="causal_params",
    category="Causal",
    passed=causal_params > 100_000_000,
    score=causal_params / 1e6,
    threshold=100,
    details=f"{format_params(causal_params)} (expected ~150M)"
))

# Causal Wave Chaining (Phase 1.5)
cwc = flx.get('causal_wave_chaining', {})
cwc_params = count_params(cwc)

report.add(TestResult(
    name="cwc_params",
    category="Causal",
    passed=cwc_params > 500_000,
    score=cwc_params / 1e3,
    threshold=500,
    details=f"{format_params(cwc_params)} (expected ~570K)"
))

In [ ]:
# Cell 14: Total Parameter Count
print(f"\n{'='*60}")
print("TOTAL PARAMETER COUNT")
print(f"{'='*60}")

# Count everything
total_params = count_params(flx)
total_tensors = count_tensors(flx)

print(f"\nTotal Parameters: {total_params:,} ({format_params(total_params)})")
print(f"Total Tensors: {total_tensors:,}")

# Category breakdown
categories = {
    'Native FLUX': count_params({k: flx[k] for k in ['cse', 'field', 'memory', 'bridges', 'causal', 
                                                      'gravity', 'thermodynamic', 'causal_wave_chaining',
                                                      'grid_to_wave', 'spatial_memory', 'adapters'] if k in flx}),
    'Embedded Models': count_params(flx.get('models', {})),
}

print(f"\nCategory Breakdown:")
for cat, params in categories.items():
    pct = 100 * params / total_params if total_params > 0 else 0
    print(f"  {cat}: {format_params(params)} ({pct:.1f}%)")

report.add(TestResult(
    name="total_params",
    category="Total",
    passed=total_params > 8_000_000_000,
    score=total_params / 1e9,
    threshold=8.0,
    details=f"{total_params:,} params (~{total_params/1e9:.2f}B)"
))

In [ ]:
# Cell 15: Validation Summary
print(f"\n{'='*60}")
print("VALIDATION SUMMARY")
print(f"{'='*60}")

report.summary()

# Final verdict
passed = sum(1 for r in report.results if r.passed)
total = len(report.results)

print(f"\n{'='*60}")
if passed == total:
    print("🎉 ALL TESTS PASSED — Flux-Apex-V1.flx is valid!")
elif passed >= total * 0.8:
    print(f"✅ MOSTLY VALID — {passed}/{total} tests passed")
else:
    print(f"⚠️ ISSUES DETECTED — {passed}/{total} tests passed")
print(f"{'='*60}")

In [ ]:
# Cell 16: Save Report
import json

output_dir = ROOT / 'output'
output_dir.mkdir(exist_ok=True)

# JSON report
report_dict = {
    'timestamp': datetime.now().isoformat(),
    'model_version': flx.get('version', 'unknown'),
    'total_params': total_params,
    'total_tensors': total_tensors,
    'results': [
        {
            'name': r.name,
            'category': r.category,
            'passed': r.passed,
            'score': r.score,
            'threshold': r.threshold,
            'details': r.details,
        }
        for r in report.results
    ]
}

report_path = output_dir / 'apex_validation_report.json'
with open(report_path, 'w') as f:
    json.dump(report_dict, f, indent=2)

print(f"✓ Report saved: {report_path}")

# Cleanup
del flx
gc.collect()
print("✓ Memory cleaned up")